# 配当情報取得

`stock/irbank.csv`（企業ID・URL取得済みの内国株式銘柄）を対象に、IRBANK（irbank.net）から年別の1株配当（予想／実績）を取得し、`stock/dividends.csv`を更新する。

## ローカル（Jupyter）実行専用の理由

[`notebooks/C01_IRBANK企業ID取得.ipynb`](./C01_IRBANK企業ID取得.ipynb)と同じ理由（IRBANKがGitHub ActionsのIPアドレスを一律ブロックしている）で、このnotebookもGitHub Actions化せず、自分のPC上でJupyterから実行する運用にしている。

## 旧Notebook（`past/`）からの変更点

- 入力を`Data/01_IDmap.csv`ではなく、現行アプリの`stock/irbank.csv`（`notebooks/C01_IRBANK企業ID取得.ipynb`が生成）に変更した。EID・URLの取得と管理はC01に一本化されているため、対象選定はC01の結果（`status=ok`かつURLがある銘柄）に従う。
- 保存形式を「銘柄コード×年の横持ち表」（`Data/03_haitou.csv`）から、他の移行済みデータ（`labels.csv`・`holdings.csv`等）と同じ「1行=1レコード」の縦持ち形式（`stock/dividends.csv`。列：`code, year, amount, is_forecast, updated_at`）に変更した。列（年）の増減が発生せず、キー名ベースでの読み書きに統一できる。この変更に伴い、旧実装にあった「直近10年分だけ保持する」列トリミング処理（`normalize_year_columns`）は不要になった（縦持ちでは列が増え続ける心配が無いため、全期間の履歴をそのまま保持する）。
- 1株配当の抽出ロジック（「一株配当」のdlタグを探す・年月と予想/実績を判定する・年ごとに実績優先で最良値を選ぶ）は旧実装をそのまま踏襲している。

## 既知の制約（旧実装から引き継いだ挙動）

無配当・配当情報が見つからなかった銘柄は`stock/dividends.csv`に何も書き込まれない。そのため「取得済み（一度でも行が書き込まれた）」の判定基準では、無配当銘柄は毎回未取得として扱われ、実行するたびに再アクセスの対象になる（旧`past/`実装から変わっていない挙動）。無配当銘柄を明示的に「確認済み」として記録し再アクセスを避ける仕組みは、必要になった時点で別途検討する。

## 実行後にすること

更新された`app_data/stock/dividends.csv`を、データリポジトリ（`palmelo2nd/app_data`）に自分でコミット・pushする（このnotebookは保存のみ行い、git操作はしない）。

In [1]:
import random
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 設定

In [2]:
# データリポジトリのローカルパス（notebooks/ から見た相対パス。app/stock/notebooks/ -> app_data/stock/）
DATA_DIR = Path("../../../app_data/stock")
IRBANK_CSV = DATA_DIR / "irbank.csv"       # 入力：企業ID・URL（notebooks/C01_IRBANK企業ID取得.ipynbが生成）
DIVIDENDS_CSV = DATA_DIR / "dividends.csv"  # 出力：年別1株配当（縦持ち）

USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
ACCEPT_HEADER = "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"

SLEEP_SEC = 1.2  # 銘柄ごとの取得間隔（秒）。IRBANK側への負荷配慮
SAVE_EVERY = 50  # この件数処理するごとに逐次保存する

JST = timezone(timedelta(hours=9))
DIVIDEND_COLUMNS = ["code", "year", "amount", "is_forecast", "updated_at"]

# True にすると、既に取得済み（dividends.csvに行がある）銘柄も含めて全件を再取得する
# （年1回程度、予想値が実績値に更新されていないかを総点検する用途を想定。旧notebookのFORCE_REFRESH_ALL相当）
FORCE_REFRESH_ALL = False

# テスト用：Noneなら全件、数値を入れるとその件数だけ処理して様子を見られる
TEST_LIMIT = None

## 関数定義

In [3]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": ACCEPT_HEADER})
    return session


def fetch_html(url: str, session: requests.Session, timeout: float = 20.0) -> str:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"
    return response.text


def polite_sleep(base: float) -> None:
    """次のリクエストまでジッター付きで待機する（IRBANK側への負荷配慮）。"""
    time.sleep(max(0.0, base + random.uniform(-0.5 * base, 0.5 * base)))


def _normalize_space(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t\r\n]+", " ", s)
    return s.strip()


def _parse_year_month_from_dt(dt_text: str) -> tuple[int, int] | None:
    t = _normalize_space(dt_text)
    m = re.search(r"(\d{4})/(\d{2})", t)
    if m:
        return int(m.group(1)), int(m.group(2))
    m2 = re.search(r"(\d{2})/(\d{2})", t)
    if m2:
        return 2000 + int(m2.group(1)), int(m2.group(2))
    return None


def _extract_amount_from_dd(dd_text: str) -> str | None:
    t = _normalize_space(dd_text).replace("円", "").replace("*", "")
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    return m.group(1) if m else None


def find_dividend_dl(soup: BeautifulSoup):
    """「一株配当」のdl.gdlを返す（idは銘柄によって別項目になるため信用せず、h2の文言で特定する）。"""
    h2 = soup.find(lambda tag: tag.name == "h2" and "一株配当" in tag.get_text(" ", strip=True))
    if h2:
        container = h2.find_parent("div")
        if container:
            dl = container.find("dl", class_="gdl")
            if dl:
                return dl
    # どうしても見つからない場合のみ、idで探す（後段の「円」ガードで誤検知を防ぐ）
    div = soup.find("div", id="c_23")
    if div:
        dl = div.find("dl", class_="gdl")
        if dl:
            return dl
    return None


def parse_dividends_yearly(html: str) -> dict[int, tuple[str, bool]]:
    """IRBANK企業ページのHTMLから、年 -> (金額文字列, 予想かどうか) の辞書を返す（年ごとに実績優先・同条件なら最新月優先）。"""
    soup = BeautifulSoup(html, "html.parser")
    dl = find_dividend_dl(soup)
    if dl is None:
        return {}

    items: list[tuple[int, int, bool, str]] = []
    current_dt_text: str | None = None
    for child in dl.find_all(["dt", "dd"], recursive=False):
        if child.name == "dt":
            current_dt_text = child.get_text(" ", strip=True)
            continue
        if child.name == "dd" and current_dt_text:
            ym = _parse_year_month_from_dt(current_dt_text)
            if ym is None:
                current_dt_text = None
                continue
            year, month = ym
            is_forecast = "予" in current_dt_text
            dd_raw = child.get_text(" ", strip=True)
            current_dt_text = None
            if "円" not in dd_raw:  # %系の誤検知（配当利回り等）を排除
                continue
            amount = _extract_amount_from_dd(dd_raw)
            if amount is None:
                continue
            items.append((year, month, is_forecast, amount))

    best: dict[int, tuple[int, bool, str]] = {}
    for year, month, is_forecast, amount in items:
        if year not in best:
            best[year] = (month, is_forecast, amount)
            continue
        cur_month, cur_is_forecast, _ = best[year]
        if cur_is_forecast and not is_forecast:
            best[year] = (month, is_forecast, amount)  # 実績が予想を上書き
        elif (not cur_is_forecast) and is_forecast:
            continue  # 実績を予想で退行させない
        elif month >= cur_month:
            best[year] = (month, is_forecast, amount)

    return {year: (amount, is_forecast) for year, (_, is_forecast, amount) in best.items()}


def load_existing(dividends_csv: Path) -> dict[tuple[str, int], dict]:
    """既存のdividends.csvを(code, year) -> 行辞書のマップとして読み込む（無ければ空）。"""
    if not dividends_csv.exists():
        return {}
    df = pd.read_csv(dividends_csv, dtype=str).fillna("")
    out = {}
    for _, row in df.iterrows():
        key = (str(row["code"]), int(row["year"]))
        out[key] = row.to_dict()
    return out


def save_dividends_csv(rows: dict[tuple[str, int], dict], dividends_csv: Path) -> None:
    dividends_csv.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(list(rows.values()), columns=DIVIDEND_COLUMNS)
    df = df.sort_values(["code", "year"])
    df.to_csv(dividends_csv, index=False)


def upsert_year_row(rows: dict[tuple[str, int], dict], code: str, year: int, amount: str, is_forecast: bool, now: str) -> None:
    """(code, year)の行を追加・更新する。既存が実績（is_forecast=False）で新しい値が予想の場合は退行させない。"""
    key = (code, year)
    existing = rows.get(key)
    if existing is not None and existing.get("is_forecast") == "False" and is_forecast:
        return
    rows[key] = {"code": code, "year": str(year), "amount": amount, "is_forecast": str(is_forecast), "updated_at": now}

## 対象銘柄の読み込み

`stock/irbank.csv`から`status=ok`かつURLがある銘柄を対象にする。`FORCE_REFRESH_ALL=False`の場合、既に`stock/dividends.csv`に1件でも行がある銘柄は除外する（無配当銘柄の再アクセスに関する制約は上記「既知の制約」参照）。

In [4]:
irbank_df = pd.read_csv(IRBANK_CSV, dtype=str).fillna("")
target_df = irbank_df[(irbank_df["status"] == "ok") & (irbank_df["url"] != "")]
all_target_codes = target_df.set_index("code")["url"].to_dict()

existing = load_existing(DIVIDENDS_CSV)
fetched_codes = {code for (code, _year) in existing.keys()}

if FORCE_REFRESH_ALL:
    target_codes = dict(all_target_codes)
else:
    target_codes = {code: url for code, url in all_target_codes.items() if code not in fetched_codes}

if TEST_LIMIT is not None:
    target_codes = dict(list(target_codes.items())[:TEST_LIMIT])

print(f"対象（IRBANK取得済み銘柄）: {len(all_target_codes)}銘柄 / 処理対象: {len(target_codes)}銘柄（FORCE_REFRESH_ALL={FORCE_REFRESH_ALL}）")

対象（IRBANK取得済み銘柄）: 3709銘柄 / 処理対象: 3709銘柄（FORCE_REFRESH_ALL=False）


## 取得実行

1銘柄の失敗（例外含む）は記録するだけで処理を継続する。`SAVE_EVERY`件ごとに`stock/dividends.csv`へ逐次保存するので、途中でカーネルを止めてもそこまでの結果は残る（再実行すれば続きから再開する）。

In [5]:
session = make_session()
succeeded = 0
no_dividend = []
failed = []

codes_list = list(target_codes.items())
for i, (code, url) in enumerate(codes_list):
    now = datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")
    try:
        html = fetch_html(url, session)
        year_to_cell = parse_dividends_yearly(html)
    except Exception as e:
        print(f"[エラー] コード: {code}（{e}）")
        failed.append(code)
        if i < len(codes_list) - 1:
            polite_sleep(SLEEP_SEC)
        continue

    if not year_to_cell:
        print(f"[無配当/情報なし] コード: {code}")
        no_dividend.append(code)
    else:
        for year, (amount, is_forecast) in year_to_cell.items():
            upsert_year_row(existing, code, year, amount, is_forecast, now)
        succeeded += 1
        years_str = f"{min(year_to_cell)}〜{max(year_to_cell)}"
        print(f"[取得] コード: {code} / {len(year_to_cell)}年分（{years_str}）")

    if i < len(codes_list) - 1:
        polite_sleep(SLEEP_SEC)

    if (i + 1) % SAVE_EVERY == 0:
        save_dividends_csv(existing, DIVIDENDS_CSV)
        print(f"--- 途中経過を保存しました（{i + 1}/{len(codes_list)}件処理） ---")

save_dividends_csv(existing, DIVIDENDS_CSV)
print(f"\n完了: 取得 {succeeded}件 / 無配当・情報なし {len(no_dividend)}件 / 失敗 {len(failed)}件（対象 {len(codes_list)}件中）")
if failed:
    print(f"失敗コード: {', '.join(failed)}")

[取得] コード: 1301 / 18年分（2010〜2027）
[無配当/情報なし] コード: 130A
[取得] コード: 1332 / 16年分（2010〜2027）
[取得] コード: 1333 / 16年分（2011〜2027）
[無配当/情報なし] コード: 135A
[取得] コード: 1375 / 11年分（2010〜2027）
[取得] コード: 1376 / 18年分（2010〜2027）
[取得] コード: 1377 / 18年分（2010〜2027）
[取得] コード: 1379 / 18年分（2010〜2027）
[無配当/情報なし] コード: 137A
[取得] コード: 1380 / 18年分（2010〜2027）
[取得] コード: 1381 / 18年分（2010〜2027）
[取得] コード: 1382 / 11年分（2010〜2027）
[取得] コード: 1383 / 15年分（2012〜2026）
[取得] コード: 1384 / 18年分（2010〜2027）
[取得] コード: 138A / 3年分（2024〜2026）
[取得] コード: 1401 / 6年分（2021〜2026）
[取得] コード: 1407 / 17年分（2010〜2026）
[取得] コード: 1414 / 18年分（2010〜2027）
[取得] コード: 1417 / 17年分（2011〜2027）
[取得] コード: 1418 / 15年分（2013〜2027）
[取得] コード: 1419 / 16年分（2012〜2027）
[取得] コード: 141A / 5年分（2023〜2027）
[取得] コード: 1420 / 14年分（2014〜2027）
[取得] コード: 1429 / 14年分（2013〜2026）
[無配当/情報なし] コード: 142A
[取得] コード: 1430 / 13年分（2015〜2027）
[取得] コード: 1431 / 12年分（2016〜2027）
[取得] コード: 1433 / 12年分（2016〜2027）
[取得] コード: 1434 / 15年分（2010〜2026）
[取得] コード: 1435 / 12年分（2012〜2026）
[取得] コード: 1436 / 12年分（2016〜2

## 保存後の確認

`stock/dividends.csv`の内容を確認したら、データリポジトリ（`app_data`）の変更を自分でコミット・pushする。

In [6]:
pd.read_csv(DIVIDENDS_CSV, dtype=str).fillna("")

,code,year,amount,is_forecast,updated_at
0,1301,2010,50,False,2026-08-23 01:49:57
1,1301,2011,50,False,2026-08-23 01:49:57
2,1301,2012,50,False,2026-08-23 01:49:57
3,1301,2013,50,False,2026-08-23 01:49:57
4,1301,2014,50,False,2026-08-23 01:49:57
...,...,...,...,...,...
47346,9997,2023,20,False,2026-08-23 03:38:37
47347,9997,2024,20.5,False,2026-08-23 03:38:37
47348,9997,2025,29,False,2026-08-23 03:38:37
47349,9997,2026,38,False,2026-08-23 03:38:37
